### Test a RAG based system with LLM being an Evaluator 🧪

Run cells in order after installing the project requirements and restarting the kernel.
Ollama must have `qwen2.5:latest` and `nomic-embed-text:v1.5`. Evaluation uses Ollama by default; set `USE_OPENAI_EVALUATOR = True` in the evaluation cell to use GPT-4o with your API key.
The sample-based Ragas API is retained for this lesson and emits deprecation warnings on Ragas 0.4.3.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

# Works when the kernel starts in the project root or a section directory.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "Section9_Testing").is_dir() and (path / "requirements.txt").is_file()
)
load_dotenv(PROJECT_ROOT / ".env")

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5:latest",
    temperature=0,
    num_predict=2048,
)


#### Embedding

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Playwright is a modern automation library for end-to-end testing. It supports multiple browsers like Chromium, Firefox, and WebKit."),
    Document(page_content="Selenium is a widely used open-source framework for web automation, supporting multiple programming languages and browsers."),
    Document(page_content="Cypress is a JavaScript-based testing tool primarily used for front-end testing. It runs in the browser and provides fast feedback loops."),
    Document(page_content="Playwright allows network interception, headless execution, and tracing for debugging complex web applications."),
    Document(page_content="Selenium WebDriver enables automated browser testing using various bindings such as Python, Java, and C#."),
    Document(page_content="Cypress has built-in support for retries, time-travel debugging, and automatic waiting, making it easy to test dynamic web pages."),
    Document(page_content="This document talks about REST API testing tools, which are unrelated to Playwright, Selenium, or Cypress.")
]

questions = [
    "What is Playwright and what browsers does it support?",
    "What is Selenium and what programming languages does it support?",
    "What is Cypress, and how is it different from Selenium?",
    "How does Playwright handle network interception and debugging?",
    "What are the key features of Selenium WebDriver?",
    "What are the main advantages of Cypress for testing?",
    "What are REST API testing tools, and how are they different from browser automation tools?"
]

# A fresh collection prevents duplicate documents when rerunning this cell.
from uuid import uuid4
vector_store = Chroma.from_documents(
    docs, embeddings, collection_name=f"testing-rag-{uuid4().hex}"
)



#### Retrieval QA and Retriever

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context. If the context is insufficient, say so."
    "\nContext:\n{context}\n\nQuestion: {question}"
)
qa_chain = prompt | llm | StrOutputParser()

query = "What does Playwright do?"
retrieved_docs = retriever.invoke(query)
response = qa_chain.invoke({
    "question": query,
    "context": "\n\n".join(doc.page_content for doc in retrieved_docs),
})
response, retrieved_docs


#### Creating a SingleTurnSample dataset for Ragas 📈


In [ ]:
dataset = []

for question, reference_doc in zip(questions, docs):
    relevant_contexts = [doc.page_content for doc in retriever.invoke(question)]
    response = qa_chain.invoke({
        "question": question,
        "context": "\n\n".join(relevant_contexts),
    })
    dataset.append({
        "user_input": question,
        "retrieved_contexts": relevant_contexts,
        "response": response,
        "reference": reference_doc.page_content,
    })

dataset


#### RAGAs Evaluation of RAG data

In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)

In [ ]:
from ragas import aevaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.run_config import RunConfig

# Set True to use GPT-4o instead; this requires OPENAI_API_KEY in .env.
USE_OPENAI_EVALUATOR = False
if USE_OPENAI_EVALUATOR:
    from langchain_openai import ChatOpenAI
    judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)
else:
    judge_llm = llm

evaluator_llm = LangchainLLMWrapper(judge_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Notebook-friendly async evaluation. Surface errors instead of hiding them as NaN.
result = await aevaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), AnswerRelevancy()],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(timeout=180, max_retries=2, max_workers=2),
    raise_exceptions=True,
)
result


In [ ]:
result.to_pandas()